# Experiment 06
## Ethics and Responsible AI Module

**Course:** Automation & Robotics / Deep Learning Lab  
**Aim:** To study and implement concepts of Responsible AI including:
1. Bias identification in datasets
2. Fairness evaluation
3. Explainable AI (XAI) visualization
4. Responsible AI case studies
5. Ethical implications in autonomous systems

### Cell 1: Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("All libraries imported successfully!")

### Cell 2: Create a Small Loan Approval Dataset (300 Samples)

We create a small realistic dataset of 300 applicants with features like `Age`, `Income`, `Credit_Score`, protected attribute `Gender`, and target `Loan_Approved`.

In [ ]:
np.random.seed(42)
n_samples = 300

# Generating synthetic data
age = np.random.randint(21, 60, n_samples)
income = np.random.randint(25000, 120000, n_samples)
credit_score = np.random.randint(500, 850, n_samples)
gender = np.random.choice(['Male', 'Female'], size=n_samples, p=[0.55, 0.45])

# Historical bias simulation: Females historically had lower approval probability in raw data
approval_prob = (credit_score / 1000) * 0.5 + (income / 150000) * 0.4
approval_prob += np.where(gender == 'Male', 0.10, -0.05) # Introducing synthetic disparity
loan_approved = (approval_prob > 0.55).astype(int)

df = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Credit_Score': credit_score,
    'Gender': gender,
    'Loan_Approved': loan_approved
})

print("Dataset Shape:", df.shape)
df.head()

### Cell 3: Bias Identification in the Dataset

Here, we check if the dataset has an unequal distribution of loan approvals across different genders (Protected Attribute).

In [ ]:
# Grouping data by gender to calculate approval rates
bias_stats = df.groupby('Gender')['Loan_Approved'].agg(['count', 'mean'])
bias_stats.columns = ['Total Applicants', 'Approval Rate']
print("--- Bias Analysis in Historical Data ---")
print(bias_stats)

# Visualizing the Bias
plt.figure(figsize=(6, 4))
sns.barplot(x='Gender', y='Loan_Approved', data=df, palette='Set2', errorbar=None)
plt.title('Loan Approval Rate by Gender (Dataset Bias)')
plt.ylabel('Approval Rate')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

male_rate = bias_stats.loc['Male', 'Approval Rate']
female_rate = bias_stats.loc['Female', 'Approval Rate']
print(f"Approval Rate for Male: {male_rate * 100:.2f}%")
print(f"Approval Rate for Female: {female_rate * 100:.2f}%")
if male_rate != female_rate:
    print(">> BIAS IDENTIFIED: Historical data favors one group over another!")

### Cell 4: Train Machine Learning Model

We train a Random Forest model on this dataset to see what the model learns.

In [ ]:
# Preprocessing: Encode Gender (Male=1, Female=0)
df_encoded = df.copy()
df_encoded['Gender'] = df_encoded['Gender'].map({'Male': 1, 'Female': 0})

X = df_encoded[['Age', 'Income', 'Credit_Score', 'Gender']]
y = df_encoded['Loan_Approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

### Cell 5: Fairness Evaluation Metrics

We evaluate fairness using standard metrics:
1. **Demographic Parity Difference:** Absolute difference between positive prediction rates for both groups.
2. **Disparate Impact Ratio:** Ratio of positive prediction rate for unprivileged group (Female) to privileged group (Male).  
*Fairness Rule (80% Rule):* Disparate Impact should ideally be between **0.80 and 1.25** to be considered fair.

In [ ]:
# Evaluate predictions across gender groups
test_results = X_test.copy()
test_results['Actual'] = y_test
test_results['Predicted'] = y_pred

# Grouping predictions by gender (1=Male, 0=Female)
male_pred_rate = test_results[test_results['Gender'] == 1]['Predicted'].mean()
female_pred_rate = test_results[test_results['Gender'] == 0]['Predicted'].mean()

print("--- Model Fairness Evaluation ---")
print(f"Model Positive Rate for Male: {male_pred_rate * 100:.2f}%")
print(f"Model Positive Rate for Female: {female_pred_rate * 100:.2f}%")

# Metric 1: Demographic Parity Difference
dp_diff = abs(male_pred_rate - female_pred_rate)
print(f"1. Demographic Parity Difference: {dp_diff:.4f}")

# Metric 2: Disparate Impact Ratio
disparate_impact = female_pred_rate / (male_pred_rate if male_pred_rate > 0 else 1)
print(f"2. Disparate Impact Ratio: {disparate_impact:.4f}")

if disparate_impact < 0.8:
    print(">> UNFAIR: Disparate impact is below 0.80 (80% Rule violated)")
else:
    print(">> FAIR: Disparate impact satisfies the 80% rule!")

### Cell 6: Explainable AI (XAI) Visualization

Explainable AI ensures that decisions are not a "black box". Here, we plot feature importance to verify which factors influenced the model's predictions the most.

In [ ]:
# Feature Importance from Random Forest
feature_names = ['Age', 'Income', 'Credit_Score', 'Gender']
importances = model.feature_importances_

feat_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plotting Explainable AI Feature Importance
plt.figure(figsize=(7, 4))
sns.barplot(x='Importance', y='Feature', data=feat_df, palette='Blues_r')
plt.title('Explainable AI (XAI): Model Feature Importance')
plt.xlabel('Relative Importance Value')
plt.ylabel('Input Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

print("--- XAI Feature Ranking ---")
for idx, row in feat_df.iterrows():
    print(f"{row['Feature']}: {row['Importance']:.4f}")

print("\nObservation: Transparent models allow auditors to verify whether sensitive attributes (e.g. Gender) unduly impact decisions.")

### Cell 7: Responsible AI Case Studies (Theory & Analysis)

#### **Case Study 1: Amazon's AI Recruitment Tool (Gender Bias)**
- **Problem:** In 2018, Amazon built an AI recruiting engine to rate job candidates' resumes from 1 to 5 stars.
- **Cause:** The model was trained on resumes submitted to the company over a 10-year period, which were predominantly from men due to historical tech industry demographics.
- **Consequence:** The model taught itself that male candidates were preferable, penalizing resumes containing the word "women's" (e.g. "women's chess club captain").
- **Outcome:** Amazon abandoned the tool and highlighted the critical need for fairness audits before deploying AI models in hiring.

---

#### **Case Study 2: COMPAS Recidivism Algorithm (Racial Bias)**
- **Problem:** COMPAS (Correctional Offender Management Profiling for Alternative Sanctions) is an algorithm used in US courts to predict the likelihood of re-offending.
- **Cause:** ProPublica investigation found the algorithm was disproportionately misclassifying African-American defendants as high risk twice as often as white defendants.
- **Consequence:** Subconscious and historical biases in criminal justice records were inherited directly by the algorithm.
- **Mitigation:** Implementing disparate impact audits, equal opportunity thresholds, and human-in-the-loop oversight.

### Cell 8: Ethical Implications in Autonomous Systems (Robotics & AI)

1. **The Moral / Trolley Dilemma in Autonomous Vehicles:**
   - In an inevitable collision scenario, how should a self-driving robot vehicle choose between protecting passengers vs pedestrians? Programmed ethical frameworks must be transparent.
2. **Safety and Fallback in Industrial Robotics:**
   - Autonomous mobile robots (AMRs) and cobots working beside human operators in factories require fail-safe sensor checks and collision prevention protocols.
3. **Accountability & Legal Liability:**
   - When an autonomous robot causes an accident, who is liable: the robot manufacturer, the software programmer, or the owner/operator?
4. **Human-in-the-Loop (HITL) Principle:**
   - High-stakes automated decisions (healthcare robotics, defence, legal) must keep a qualified human operator in the decision pipeline.

## Result and Conclusion

1. Identified historical bias in a dataset by comparing group approval rates across gender.
2. Evaluated fairness using **Demographic Parity Difference** and **Disparate Impact Ratio (80% rule)**.
3. Implemented Explainable AI (XAI) feature importance visualization to interpret model decisions transparently.
4. Documented Responsible AI case studies and ethical implications in autonomous robotics systems.